In [1]:
import pandas as pd
import numpy as np

test = pd.read_parquet('/kaggle/input/datasets/imdevshah11/ml-dataset/test_features_processed.parquet')
train = pd.read_parquet('/kaggle/input/datasets/imdevshah11/ml-dataset/train_features_cleaned.parquet')

print(train.shape)
print(train.columns.tolist())

(3195845, 111)
['id', 'code', 'sub_code', 'sub_category', 'horizon', 'ts_index', 'feature_a', 'feature_b', 'feature_c', 'feature_d', 'feature_e', 'feature_f', 'feature_g', 'feature_h', 'feature_i', 'feature_j', 'feature_k', 'feature_l', 'feature_m', 'feature_n', 'feature_o', 'feature_p', 'feature_q', 'feature_r', 'feature_s', 'feature_t', 'feature_u', 'feature_v', 'feature_w', 'feature_x', 'feature_y', 'feature_z', 'feature_aa', 'feature_ab', 'feature_ac', 'feature_ad', 'feature_ae', 'feature_af', 'feature_ag', 'feature_ah', 'feature_ai', 'feature_aj', 'feature_ak', 'feature_al', 'feature_am', 'feature_an', 'feature_ao', 'feature_ap', 'feature_aq', 'feature_ar', 'feature_as', 'feature_at', 'feature_au', 'feature_av', 'feature_aw', 'feature_ax', 'feature_ay', 'feature_az', 'feature_ba', 'feature_bb', 'feature_bc', 'feature_bd', 'feature_be', 'feature_bf', 'feature_bg', 'feature_bh', 'feature_bi', 'feature_bj', 'feature_bk', 'feature_bl', 'feature_bm', 'feature_bn', 'feature_bo', 'featur

In [3]:
import pandas as pd
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.preprocessing import RobustScaler
import warnings
warnings.filterwarnings("ignore")

# ── 1. HELPERS ──────────────────────────────────────────────────────────
def add_category_features(df, train_source):
    # This captures the 'similarities' mentioned in project tips
    for col in ['code', 'sub_category']:
        mapping = train_source.groupby(col)['y_target'].mean().to_dict()
        df[f'{col}_mean'] = df[col].map(mapping).fillna(0)
    return df

# ── 2. DATA PREP ────────────────────────────────────────────────────────
drop_cols = ['id', 'group_id', 'y_target', 'weight', 'code', 'sub_code', 'sub_category', 'ts_index']
feature_cols = [c for c in train.columns if c not in drop_cols]

# Create Similarity Features
train_df = add_category_features(train.copy(), train_source=train)
test_df = add_category_features(test.copy(), train_source=train)

final_features = feature_cols + ['code_mean', 'sub_category_mean']

# Handle missing values
train_median = train_df[final_features].median(numeric_only=True)
X_train_full = train_df.fillna(train_median)[final_features]
y_train_full = train_df['y_target']
w_train_full = train_df['weight']

# Fit Scaler
scaler = RobustScaler()
X_train_sc = scaler.fit_transform(X_train_full)

# ── 3. FIT MODEL (Alpha=500 for high noise) ───────────────────────────
print(f"Fitting final Ridge model...")
final_model = Ridge(alpha=500.0)
final_model.fit(X_train_sc, y_train_full, sample_weight=w_train_full)

# ── 4. PREDICT & PROTECT ────────────────────────────────────────────────
print("\nPreparing test predictions...")
X_test = test_df.fillna(train_median)[final_features]

# CLIPPING LOCK 1: Clip inputs based on training feature bounds
for col in final_features:
    c_min = float(np.nanmin(X_train_full[col]))
    c_max = float(np.nanmax(X_train_full[col]))
    X_test[col] = np.clip(X_test[col].values, c_min, c_max)

# Scale
X_test_sc = scaler.transform(X_test)

# Generate Raw Predictions
preds = final_model.predict(X_test_sc)

# CLIPPING LOCK 2: Clip final outputs to match training target bounds
# This ensures your predictions stay within a sane range (e.g., -0.1 to 0.1)
y_min = float(y_train_full.min())
y_max = float(y_train_full.max())
final_preds = np.clip(preds, y_min, y_max)

# ── 5. SAVE & VERIFY ────────────────────────────────────────────────────
print(f"\nTraining Target Range: [{y_min:.6f}, {y_max:.6f}]")
print(f"Prediction Range     : [{final_preds.min():.6f}, {final_preds.max():.6f}]")

submission = pd.DataFrame({'id': test['id'], 'prediction': final_preds})
submission.to_csv('submission.csv', index=False)
print("\nFinal SECURE submission saved ✅")

Fitting final Ridge model...

Preparing test predictions...

Training Target Range: [-1646.887504, 1521.665638]
Prediction Range     : [-150.253725, 157.760962]

Final SECURE submission saved ✅
